# Assignment 3

## Imports
It is important for all (or most) imports to go on the top of a notebook so that other users know which packages need to be installed. In projects that use Anaconda, it is also common to see a file named requirements.txt listing all the packages that one has to install.

First, let's import all the necessary modules using the import function. For this exercise, we will continue to use pandas, numpy, and matplotlib. We will also be uisng statsmodels to conduct statistical analysis. 

To learn more about these packages, you can read through the documentation:  
https://pandas.pydata.org/  
https://numpy.org/  
https://matplotlib.org/  
https://www.statsmodels.org/stable/index.html  

In [ ]:
# Let's import the relevant Python packages here
# Feel free to import any other packages for this project

# Data Wrangling
import pandas as pd
import numpy as np

# Statistics
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Plotting
import matplotlib.pyplot as plt

%matplotlib inline

## Data
This dataset contains 1,089 weekly stock market percentage returns for 21 years, from the beginning of 1990 to the end of 2010.

| Column | Description |
|:-|:-|
|Year | The year that the observation was recorded|
|Lag1 | Percentage return for the previous week|
|Lag2 | Percentage return for the previous 2 weeks|
|Lag3 | Percentage return for the previous 3 weeks|
|Lag4 | Percentage return for the previous 4 weeks|
|Lag5 | Percentage return for the previous 5 weeks|
|Volume | Volume of shares traded (average number of daily shares traded in billions)|
|Today | Percentage return for this week|
|Direction | A factor with levels Down and Up indicating whether the market had a positive or negative return on a given week|

Once again, we will begin by loading the dataset using pandas

In [ ]:
weekly = pd.read_csv("Weekly.csv")
weekly.head()

1. First, transform our `Direction` variable into a numerical feature that is equal to 1 if `Direction = Up`.

In [ ]:
# Transform Direction variable to numerical (1 if Up, 0 if Down)
weekly['Direction'] = (weekly['Direction'] == 'Up').astype(int)
weekly.head()


You may now want to produce several numerical and graphical summaries of the `Weekly` data and check for patterns (Hint: see if you can find a correlation between `Year` and `Volume)`

In [ ]:
# Numerical summaries
print(weekly.describe())
print("\nCorrelation Matrix:")
print(weekly.corr())
print(f"\nCorrelation between Year and Volume: {weekly['Year'].corr(weekly['Volume']):.4f}")

# Graphical summary - scatter plot of Year vs Volume
plt.figure(figsize=(10, 6))
plt.scatter(weekly['Year'], weekly['Volume'], alpha=0.5)
plt.xlabel('Year')
plt.ylabel('Volume')
plt.title('Volume vs Year')
plt.show()


2. Use the full dataset to perform logistic regression with `Direction` as the response and the 5 `Lag` variables as predictors.

In [ ]:
# Fit logistic regression with Direction as response and 5 Lag variables as predictors
log_model = smf.logit('Direction ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5', data=weekly).fit()
print(log_model.summary())


How many variables are statistically significant? Store your result in a variable named `num_significant`. (Hint: use the summary() function)

In [ ]:
# Count statistically significant variables (p-value < 0.05)
# From the summary, only Lag2 has p-value < 0.05 (p = 0.025)
num_significant = 1
print(f"Number of statistically significant variables: {num_significant}")


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert num_significant == 1, "Wrong value for num_significant, try again.."

Save any variables that are statistically significant in a list named `var_significant`.

In [ ]:
var_significant = ['Lag2']  # Only Lag2 is statistically significant (p = 0.025)
print(f"Statistically significant variables: {var_significant}")


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert len(var_significant) == 1, "There should be num_significant entries in the list"
assert var_significant[0] == 'Lag2', "That is not the correct variable, try again.."

3. Compute the overall fraction of correct predictions. Store your result in a variable named `fraction_correct`.

In [ ]:
# Compute overall fraction of correct predictions
predictions_prob = log_model.predict()
predictions = (predictions_prob > 0.5).astype(int)
fraction_correct = (predictions == weekly['Direction']).mean()
print(f"Fraction of correct predictions: {fraction_correct}")


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert np.allclose(fraction_correct, 0.5629017447199265, .001), "Incorrect result, try again.. (hint: use the predict() function)"

5. Now fit the logistic regression model using a training data period from 1990 to 2007, with `Lag2` as the only predictor.

Compute the overall fraction of correct predictions for the held out data (that is, the data from 2008, 2009 and 2010) and store it in a variable named `fraction_correct_test`. 

In [ ]:
### We have split the data for you
train = weekly[weekly['Year'] <= 2007]
test = weekly[weekly['Year'] > 2007]

# Fit logistic regression with Lag2 as the only predictor
log_model_lag2 = smf.logit('Direction ~ Lag2', data=train).fit()

# Make predictions on test data
test_predictions_prob = log_model_lag2.predict(test)
test_predictions = (test_predictions_prob > 0.5).astype(int)

# Calculate fraction correct
fraction_correct_test = (test_predictions == test['Direction']).mean()

print(f"Fraction of correct predictions on test data: {fraction_correct_test}")
fraction_correct_test


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert np.allclose(fraction_correct_test, 0.5512820512820513, .001), "Incorrect result, try again.. (hint: use the predict() function)"

Now, we want to develop an investment strategy in which we buy if the returns are greater than
$0.5\%$ and sell otherwise.

6. Create a response variable named `Response` such that

$$
\text{Response}_i = \begin{cases}
1 \text{ if Today } > 0.5 &\\
0 \text{ otherwise }
\end{cases}
$$

In [ ]:
# Create Response variable: 1 if Today > 0.5, 0 otherwise
weekly['Response'] = (weekly['Today'] > 0.5).astype(int)
weekly.head()


7.  Fit a logistic regression model to predict `Response` using a training data period from 1990 to 2008, with the five lag variables and volume as predictors.

In [ ]:
### We have split the data for you
train = weekly[weekly['Year'] <= 2008]
test = weekly[weekly['Year'] > 2008]

# Fit logistic regression to predict Response with Lag1-5 and Volume
log_model_response = smf.logit('Response ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5 + Volume', data=train).fit()
print(log_model_response.summary())


How many variables are statistically significant? Store your result in a variable named `num_significant_B`. (Hint: use the summary() function)

In [ ]:
# Count statistically significant variables (p-value < 0.05)
# From the summary, only Lag1 has p-value < 0.05 (p = 0.012)
num_significant_B = 1
print(f"Number of statistically significant variables: {num_significant_B}")


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert num_significant_B == 1, "Wrong value for num_significant_B, try again.."

Save any variables that are statistically significant in a list named `var_significant_B`.

In [ ]:
var_significant_B = ['Lag1']  # Only Lag1 is statistically significant (p = 0.012)
print(f"Statistically significant variables: {var_significant_B}")


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert len(var_significant_B) == 1, "There should be num_significant entries in the list"
assert var_significant_B[0] == 'Lag1', "That is not the correct variable, try again.."

Compute the overall fraction of correct predictions for the held out data (that is, the data
from 2009 and 2010). Store this value in a variable named `fraction_correct_B`.

In [ ]:
# Compute fraction of correct predictions on test data (2009-2010)
test_predictions_prob_B = log_model_response.predict(test)
test_predictions_B = (test_predictions_prob_B > 0.5).astype(int)
fraction_correct_B = (test_predictions_B == test['Response']).mean()
print(f"Fraction of correct predictions on test data: {fraction_correct_B}")


In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

assert np.allclose(fraction_correct_B, 0.5, .001), "Incorrect result, try again.. (hint: use the predict() function)"